# Zing Generation Demo

Generate ZINC samples from a saved `ConditionalNodeFieldGraphGenerator`.

- inspect saved checkpoints
- load one model
- sample with and without feasibility filtering


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

from pathlib import Path
import os
import random

os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

import numpy as np
from IPython.core.display import HTML

HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell; text-align: center; vertical-align: middle;}</style>')

from conditional_node_field_graph_generator.notebooks import configure_notebook
globals().update(configure_notebook(require_nsppk=True, print_torch=True))

from abstractgraph_graphicalizer.chem import draw_molecules
from conditional_node_field_graph_generator.extensions.demo import show_molecules
from conditional_node_field_graph_generator.persistence import (
    list_saved_graph_generators,
    load_graph_generator,
)


In [ ]:
import sys
import time
from types import MethodType

from sklearn.linear_model import SGDClassifier, SGDRegressor
from nsppk import NSPPK

from abstractgraph.operators import (
    add,
    combination,
    compose,
    connected_component,
    cycle,
    edge,
    filter_by_edge_label,
    merge,
    neighborhood,
    tree,
    unlabel,
)
from abstractgraph_graphicalizer.chem import ZINCLoader, draw_molecules as display_graphs
from abstractgraph_ml.estimators import GraphEstimator
from abstractgraph_ml.feasibility import (
    FeasibilityEstimator,
    FeasibilityEstimatorFeatureCannotExist,
)

EDGE_GENERATIVE_SRC = Path('/home/fabrizio/code/abstractgraph-ecosystem/repos/abstractgraph-generative/src')
if not EDGE_GENERATIVE_SRC.exists():
    raise RuntimeError(f'abstractgraph-generative checkout not found at {EDGE_GENERATIVE_SRC}')
if str(EDGE_GENERATIVE_SRC) not in sys.path:
    sys.path.insert(0, str(EDGE_GENERATIVE_SRC))

from abstractgraph_generative.edge_generator import EdgeGenerator

def _find_early_stop_in_beam_any_phase(
    self,
    beam,
    *,
    scored,
    n_edges: int,
    target,
    target_lambda: float,
    fallback_index: int,
    graph_index: int,
    total_phases: int,
    start_time: float,
    verbose: bool,
):
    if not self.early_stop_if_final_feasible or not beam:
        return None

    current_states = self._score_current_beam_states_for_early_stop(
        beam,
        target=target,
        target_lambda=target_lambda,
        fallback_index=fallback_index,
    )
    if not current_states:
        return None

    best_current = current_states[0]
    best_expansion_score = None
    if scored['feasible_candidates']:
        best_expansion_score = scored['feasible_candidates'][0].get(
            'selection_score',
            scored['feasible_candidates'][0]['score'],
        )

    current_score = best_current.get('selection_score', best_current['score'])
    if best_expansion_score is not None and current_score < best_expansion_score:
        return None

    path = self._reconstruct_path(best_current)
    if verbose:
        elapsed_str = self._format_minutes_seconds(time.perf_counter() - start_time)
        current_edges = best_current['graph'].number_of_edges()
        edge_shortfall = max(0, n_edges - current_edges)
        print(
            f'[graph {graph_index}] early_stop phase={fallback_index + 2}/{total_phases} '
            f"depth={best_current['depth']} max_depth={self.max_depth_} "
            f'edges={current_edges} edge_shortfall={edge_shortfall} remaining_edges=0 '
            f'tried={self.n_tried_} elapsed={elapsed_str} eta=0m 0.0s'
        )
        print(
            f'[graph {graph_index}] early_stop_selection_score='
            f"{current_score:.3f} best_expansion_selection_score="
            f'{self._format_optional_score(best_expansion_score)}'
        )
    return path

def repair_candidate_graphs(
    candidate_graphs,
    *,
    graph_generator,
    model_filename,
    notebook_data_root,
    random_seed,
    fallback_dataset='zinc_250k',
    store_per_size=256,
    n_neighbors=12,
    draw_graphs_fn=None,
    show_summary=True,
):
    if graph_generator.feasibility_estimator is None:
        raise RuntimeError('EdgeGenerator repair requires a fitted feasibility estimator.')

    sample_node_counts = sorted({graph.number_of_nodes() for graph in candidate_graphs})
    load_limit = max(1000, store_per_size * 4)
    edge_repair_dataset = model_filename.split('-', 1)[0]
    edge_repair_data_root = notebook_data_root / 'zinc'

    loader = ZINCLoader(root=edge_repair_data_root, on_error='skip')
    stored_counts = {node_count: 0 for node_count in sample_node_counts}
    edge_repair_graphs = []
    edge_repair_dataset_used = {}
    for node_count in sample_node_counts:
        selected_graphs = []
        for dataset_name in (edge_repair_dataset, fallback_dataset):
            candidate_store_graphs, _ = loader.load(
                dataset_name,
                limit=load_limit,
                min_node_count=node_count,
                max_node_count=node_count,
            )
            selected_graphs = candidate_store_graphs[:store_per_size]
            if selected_graphs:
                edge_repair_dataset_used[node_count] = dataset_name
                break
        edge_repair_graphs.extend(selected_graphs)
        stored_counts[node_count] = len(selected_graphs)

    missing_sizes = [node_count for node_count, count in stored_counts.items() if count == 0]
    if missing_sizes:
        raise RuntimeError(
            f'Could not load ZINC repair graphs for node counts: {missing_sizes} '
            f'from datasets {[edge_repair_dataset, fallback_dataset]}'
        )

    feasibility_kwargs = dict(
        nbits=19,
        parallel=True,
        backend='loky',
        n_jobs=-1,
    )
    partial_feasibility_estimators = [
        FeasibilityEstimatorFeatureCannotExist(
            decomposition_function=compose(neighborhood(radius=2), unlabel()),
            **feasibility_kwargs,
        ),
        FeasibilityEstimatorFeatureCannotExist(
            decomposition_function=neighborhood(radius=1),
            **feasibility_kwargs,
        ),
        FeasibilityEstimatorFeatureCannotExist(
            decomposition_function=cycle(),
            **feasibility_kwargs,
        ),
    ]
    partial_feasibility_estimator = FeasibilityEstimator(partial_feasibility_estimators)

    final_feasibility_estimators = [
        FeasibilityEstimatorFeatureCannotExist(
            decomposition_function=compose(neighborhood(radius=2), unlabel()),
            **feasibility_kwargs,
        ),
        FeasibilityEstimatorFeatureCannotExist(
            decomposition_function=neighborhood(radius=1),
            **feasibility_kwargs,
        ),
        FeasibilityEstimatorFeatureCannotExist(
            decomposition_function=compose(
                connected_component(),
                unlabel(),
                merge(use_edges=True),
                filter_by_edge_label(must_have_one_of=['aromatic']),
                edge(),
            ),
            **feasibility_kwargs,
        ),
        FeasibilityEstimatorFeatureCannotExist(
            decomposition_function=cycle(),
            **feasibility_kwargs,
        ),
        FeasibilityEstimatorFeatureCannotExist(
            decomposition_function=compose(
                combination(number_of_elements=2, distance=0),
                cycle(),
                unlabel(),
            ),
            **feasibility_kwargs,
        ),
    ]
    final_feasibility_estimator = FeasibilityEstimator(final_feasibility_estimators)

    use_sparse_linear_mode = True
    vectorizer_kwargs = dict(
        radius=1,
        distance=4,
        connector=1,
        nbits=14,
        parallel=True,
    )

    graph_transformer = NSPPK(**vectorizer_kwargs, dense=not use_sparse_linear_mode)
    graph_estimator_model = SGDClassifier(
        loss='log_loss',
        alpha=1e-4,
        max_iter=1000,
        tol=1e-3,
        random_state=random_seed,
        class_weight='balanced',
    )
    target_estimator_model = SGDRegressor(
        loss='epsilon_insensitive',
        alpha=1e-4,
        max_iter=1000,
        tol=1e-3,
        random_state=random_seed,
    )
    edge_risk_estimator_model = SGDRegressor(
        loss='epsilon_insensitive',
        alpha=1e-4,
        max_iter=1000,
        tol=1e-3,
        random_state=random_seed,
    )

    graph_estimator = GraphEstimator(
        transformer=graph_transformer,
        estimator=graph_estimator_model,
    )
    target_estimator = None
    edge_risk_estimator = GraphEstimator(
        transformer=NSPPK(**vectorizer_kwargs, dense=not use_sparse_linear_mode),
        estimator=edge_risk_estimator_model,
    )

    edge_repair_generator = EdgeGenerator(
        partial_feasibility_estimator=partial_feasibility_estimator,
        final_feasibility_estimator=final_feasibility_estimator,
        graph_estimator=graph_estimator,
        target_estimator=target_estimator,
        edge_risk_estimator=edge_risk_estimator,
        target_estimator_mode='regression',
        decomposition_function=add(cycle(), tree()),
        enforce_diversity=False,
        n_negative_per_positive=5,
        n_replicates=5,
        beam_size=2,
        max_restarts=4,
        fit_n_jobs=-1,
        fit_backend='loky',
        edge_risk_lambda=0.25,
        verbose=True,
        seed=random_seed,
        early_stop_if_final_feasible=True,
        require_single_connected_component=True,
    )
    edge_repair_generator._find_early_stop_in_beam = MethodType(
        _find_early_stop_in_beam_any_phase,
        edge_repair_generator,
    )
    edge_repair_generator.store(edge_repair_graphs)

    if draw_graphs_fn is None:
        draw_graphs_fn = lambda graphs, **kwargs: display_graphs(
            graphs,
            n_graphs_per_line=7,
            **kwargs,
        )

    sample_feasible = np.asarray(
        graph_generator.feasibility_estimator.predict(candidate_graphs),
        dtype=bool,
    )
    repaired_graphs = []
    for sample_idx, graph in enumerate(candidate_graphs):
        repaired_graph = edge_repair_generator.repair(
            graph,
            n_neighbors=min(n_neighbors, len(edge_repair_graphs)),
            draw_graphs_fn=draw_graphs_fn,
            return_path=False,
        )
        repaired_graphs.append(repaired_graph)
        print(
            f'sample {sample_idx}: nodes={graph.number_of_nodes()} edges={graph.number_of_edges()} '
            f'feasible_before={bool(sample_feasible[sample_idx])} repaired={repaired_graph is not None}'
        )

    valid_repaired_graphs = [graph for graph in repaired_graphs if graph is not None]
    repaired_feasible = None
    if valid_repaired_graphs:
        repaired_feasible = np.asarray(
            graph_generator.feasibility_estimator.predict(valid_repaired_graphs),
            dtype=bool,
        )

    if show_summary:
        if valid_repaired_graphs:
            print('edge_repair_dataset_used =', edge_repair_dataset_used)
            print('stored_counts =', stored_counts)
            print('repaired_feasible =', repaired_feasible.tolist())
            show_molecules(
                candidate_graphs,
                n=len(candidate_graphs),
                title='Raw samples before EdgeGenerator repair',
            )
            show_molecules(
                valid_repaired_graphs,
                n=len(valid_repaired_graphs),
                title='Raw samples after EdgeGenerator repair',
            )
        else:
            print('EdgeGenerator repair did not return any repaired samples.')

    return {
        'repaired_graphs': repaired_graphs,
        'valid_repaired_graphs': valid_repaired_graphs,
        'sample_feasible': sample_feasible,
        'repaired_feasible': repaired_feasible,
        'edge_repair_dataset_used': edge_repair_dataset_used,
        'stored_counts': stored_counts,
        'edge_repair_generator': edge_repair_generator,
        'edge_repair_graphs': edge_repair_graphs,
    }


In [ ]:
MODEL_FILENAME = 'zinc15-nonstreaming-d64-s0-99-b128-e350.pkl'
#MODEL_FILENAME = 'zinc15-streaming-d64-s0-5-w256-b8-e256.pkl'
#MODEL_FILENAME = 'zinc18-streaming-d64-s0-9-w2049-b128-e256.pkl'
MODEL_FILENAME = 'zinc20-nonstreaming-d64-s0-99-b128-e350.pkl'
N_SAMPLES = 2

DECODER_VERBOSE = 1
DECODER_EXISTENCE_THRESHOLD = 0.5
DECODER_ENFORCE_CONNECTIVITY = True
DECODER_DEGREE_SLACK_PENALTY = 1e6
DECODER_WARM_START_MST = True
DECODER_N_JOBS = -1
DECODER_GRAPH_RENDERER = draw_molecules

USE_FEASIBILITY_FILTERING = True
MAX_FEASIBILITY_ATTEMPTS = 5
FEASIBILITY_CANDIDATES_PER_ATTEMPT = 2
FEASIBILITY_FAILURE_MODE = 'return_partial'
MAX_FEASIBILITY_SECONDS_PER_SAMPLE = 10.0
FEASIBILITY_ORACLE_CANDIDATES_PER_ATTEMPT = 1
MAX_ORACLE_ITERATIONS = 4
ORACLE_USE_NODE_LABEL_CUTS = True
ORACLE_USE_EDGE_LABEL_CUTS = True
RANDOM_SEED = 7
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [ ]:
list_saved_graph_generators(SAVED_GENERATOR_ROOT)

graph_generator = load_graph_generator(MODEL_FILENAME, model_dir=SAVED_GENERATOR_ROOT)
graph_generator.graph_decoder.verbose = DECODER_VERBOSE
graph_generator.graph_decoder.existence_threshold = DECODER_EXISTENCE_THRESHOLD
graph_generator.graph_decoder.enforce_connectivity = DECODER_ENFORCE_CONNECTIVITY
graph_generator.graph_decoder.degree_slack_penalty = DECODER_DEGREE_SLACK_PENALTY
graph_generator.graph_decoder.warm_start_mst = DECODER_WARM_START_MST
graph_generator.graph_decoder.n_jobs = DECODER_N_JOBS
graph_generator.graph_decoder.diagnostic_graph_renderer = DECODER_GRAPH_RENDERER
graph_generator.use_feasibility_filtering = USE_FEASIBILITY_FILTERING
graph_generator.max_feasibility_attempts = MAX_FEASIBILITY_ATTEMPTS
graph_generator.feasibility_candidates_per_attempt = FEASIBILITY_CANDIDATES_PER_ATTEMPT
graph_generator.feasibility_failure_mode = FEASIBILITY_FAILURE_MODE
graph_generator.max_feasibility_seconds_per_sample = MAX_FEASIBILITY_SECONDS_PER_SAMPLE
graph_generator.feasibility_oracle_candidates_per_attempt = FEASIBILITY_ORACLE_CANDIDATES_PER_ATTEMPT
graph_generator.max_oracle_iterations = MAX_ORACLE_ITERATIONS
graph_generator.oracle_use_node_label_cuts = ORACLE_USE_NODE_LABEL_CUTS
graph_generator.oracle_use_edge_label_cuts = ORACLE_USE_EDGE_LABEL_CUTS


In [ ]:
print('Loaded graph generator =', getattr(graph_generator, 'model_name', MODEL_FILENAME))
print('Decoder verbose =', graph_generator.graph_decoder.verbose)
print('Decoder existence_threshold =', graph_generator.graph_decoder.existence_threshold)
print('Decoder enforce_connectivity =', graph_generator.graph_decoder.enforce_connectivity)
print('Decoder degree_slack_penalty =', graph_generator.graph_decoder.degree_slack_penalty)
print('Decoder warm_start_mst =', graph_generator.graph_decoder.warm_start_mst)
print('Decoder n_jobs =', graph_generator.graph_decoder.n_jobs)
print('Decoder graph renderer =', getattr(graph_generator.graph_decoder.diagnostic_graph_renderer, '__name__', graph_generator.graph_decoder.diagnostic_graph_renderer))
print('Feasibility estimator available =', graph_generator.feasibility_estimator is not None)
print('use_feasibility_filtering =', graph_generator.use_feasibility_filtering)
print('max_feasibility_attempts =', graph_generator.max_feasibility_attempts)
print('feasibility_candidates_per_attempt =', graph_generator.feasibility_candidates_per_attempt)
print('feasibility_failure_mode =', graph_generator.feasibility_failure_mode)
print('max_feasibility_seconds_per_sample =', graph_generator.max_feasibility_seconds_per_sample)
print('feasibility_oracle_candidates_per_attempt =', graph_generator.feasibility_oracle_candidates_per_attempt)
print('max_oracle_iterations =', graph_generator.max_oracle_iterations)
print('oracle_use_node_label_cuts =', graph_generator.oracle_use_node_label_cuts)
print('oracle_use_edge_label_cuts =', graph_generator.oracle_use_edge_label_cuts)


In [ ]:
raw_samples = graph_generator.sample(
    n_samples=N_SAMPLES,
    apply_feasibility_filtering=False,
)
show_molecules(raw_samples, n=N_SAMPLES, title='Zing generation samples without feasibility filtering')


In [ ]:
repair_results = repair_candidate_graphs(
    raw_samples,
    n_neighbors=14,
    graph_generator=graph_generator,
    model_filename=MODEL_FILENAME,
    notebook_data_root=NOTEBOOK_DATA_ROOT,
    random_seed=RANDOM_SEED,
)


In [ ]:
if graph_generator.feasibility_estimator is None:
    raise RuntimeError('Feasibility estimator is unavailable in this environment.')

graph_generator.feasibility_rejection_mode = "fallback_unfiltered" #"fallback_unfiltered"  or "strict"

filtered_samples = graph_generator.sample(
    n_samples=N_SAMPLES,
    apply_feasibility_filtering=True,
)
show_molecules(filtered_samples, n=N_SAMPLES, title='Zing generation samples with feasibility filtering')


In [ ]:
repair_results = repair_candidate_graphs(
    filtered_samples,
    graph_generator=graph_generator,
    model_filename=MODEL_FILENAME,
    notebook_data_root=NOTEBOOK_DATA_ROOT,
    random_seed=RANDOM_SEED,
)
